In [2]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

repo_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(repo_root))

from tutorial._infra import torch_frontend as torch_nb

torch_nb.print_setup()

class ConvAddRelu(nn.Module):
    def __init__(self, cin=3, cout=8, k=3):
        super().__init__()
        self.conv = nn.Conv2d(cin, cout, k, padding=k // 2, bias=False)
        self.bias_vec = nn.Parameter(torch.zeros(cout))

    def forward(self, x):
        y = self.conv(x)
        y = y + self.bias_vec.view(1, -1, 1, 1)
        return F.relu(y)


Repository root -> /Users/admin/Programming/ESWEEK-tutorial/Cinnamon
torch-mlir-opt  -> /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/.venv/bin/torch-mlir-opt
Artifacts dir   -> /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/tutorial/artifacts


In [3]:
import torch
from torch_mlir import fx

conv_module = ConvAddRelu().eval()
example_input = torch.randn(1, 3, 16, 16, dtype=torch.float32)

torch_module = fx.export_and_import(conv_module, example_input, func_name="main")
torch_ir = torch_module.operation.get_asm()

torch_file = torch_nb.ARTIFACTS_DIR / "conv_add_relu_torch.mlir"
torch_file.write_text(torch_ir)

print(f"Wrote Torch dialect IR → {torch_file.resolve()}")


ImportError: cannot import name 'ir' from 'torch_mlir._mlir_libs._mlir' (unknown location)